# Data Cleaning — Kosovo Remittances Channels Study
Reads raw xlsx files, translates Albanian labels to English, normalises to tidy long format, saves as parquet.

In [153]:
import pandas as pd
import numpy as np
import re
import warnings
from pathlib import Path
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 12)
pd.set_option('display.width', 130)

RAW_KS  = Path('raw_data/kosovo')
RAW_BLK = Path('raw_data/balkans')
OUT_DIR = Path('parquet')
OUT_DIR.mkdir(exist_ok=True)

In [2]:
# ── Shared translation dictionaries ──────────────────────────────────────────

MONTH_AL = {
    'Janar':1, 'Shkurt':2, 'Mars':3, 'Prill':4, 'Maj':5, 'Qershor':6,
    'Korrik':7, 'Gusht':8, 'Shtator':9, 'Tetor':10, 'Nëntor':11, 'Nentor':11, 'Dhjetor':12,
    'Jan':1,'Feb':2,'Mar':3,'Apr':4,'May':5,'Jun':6,'Jul':7,'Aug':8,'Sep':9,'Oct':10,'Nov':11,'Dec':12,
}
MONTH_EN = {v: k for k, v in {
    1:'January',2:'February',3:'March',4:'April',5:'May',6:'June',
    7:'July',8:'August',9:'September',10:'October',11:'November',12:'December'
}.items()}

QUARTER_MAP = {'TM1':'Q1','TM2':'Q2','TM3':'Q3','TM4':'Q4'}

LABEL = {
    # General
    'Gjithsej':'Total','Gjithsej ':'Total','Meshkuj':'Male','Mashkull':'Male',
    'Femra':'Female','Femër':'Female','Nuk dihet':'Unknown',
    'Viti':'Year','Muaji':'Month','Tremujori':'Quarter',
    'Zerat/Vitet':'Item','Përshkrimi':'Description','Baza vjetore':'Annual basis',
    # BOP / remittances
    'Bilanci nga llogaria rrjedhëse dhe kapitale':'Current and capital account balance',
    'Llogaria rrjedhëse':'Current account',
    'Mallrat':'Goods','Shërbimet':'Services','Bilanci':'Balance',
    'Dërgesat e emigrantëve sipas kanaleve':'Remittances by channel',
    'Gjithsej dërgesat e migrantëv':'Total remittances',
    'Gjithsej dërgesat e migrantëve':'Total remittances',
    'Bankat':'Banks','Agjencitë për transfer të mjeteve':'Transfer agencies',
    'Posta':'Post office','Dorë në dorë':'Hand to hand (cash)',
    'Të tjera':'Other','Remitencat e punëtorve':'Worker remittances',
    'Kompensimi i punëtorve':'Compensation of employees',
    'Të ardhurat nga investimet':'Investment income',
    'Investimet direkte':'Direct investment','Investimet portfolio':'Portfolio investment',
    'Investimet tjera':'Other investment','Llogaria kapitale':'Capital account',
    'Llogaria financiare':'Financial account',
    'Investimet e huaja direkte':'Foreign direct investment (net)',
    'Investimet e tjera':'Other investment (fin. account)',
    'Rezervat':'Reserve assets',
    # GDP
    'Bruto Vlera e Shtuar (BVSH)':'Gross Value Added (GVA)',
    'Bujqësia, gjuetia , pylltaria dhe peshkimi':'Agriculture, hunting, forestry and fishing',
    'Bujqësia, pylltaria dhe peshkimi':'Agriculture, forestry and fishing',
    'Industria nxjerrëse':'Mining and quarrying','Industria përpunuese':'Manufacturing',
    'Furnizimi me energji elektrike, me gaz':'Electricity and gas supply',
    'Furnizimi me energji elektrike dhe gaz':'Electricity and gas supply',
    'Furnizimi me ujë':'Water supply','Ndërtimtaria':'Construction',
    'Tregtia me shumicë dhe pakicë':'Wholesale and retail trade',
    'Transporti dhe magazinimi':'Transportation and storage',
    'Akomodimi dhe shërbimi ushqimor':'Accommodation and food services',
    'Informimi dhe komunikimi':'Information and communication',
    'Aktivitetet financiare dhe të sigurimit':'Financial and insurance activities',
    'Aktivitetet e pasurive të patundshme':'Real estate activities',
    'Aktivitetet profesionale, shkencore dhe teknike':'Professional and scientific activities',
    'Aktivitetet administrative dhe shërbimeve mbështetëse':'Administrative and support activities',
    'Administrata publike dhe mbrojtja':'Public administration and defense',
    'Arsimi':'Education','Aktivitetet e kujdesit shëndetësor':'Human health activities',
    'Aktivitetet e tjera shërbyese':'Other service activities',
    'Tatimet neto mbi produktet':'Net taxes on products',
    'Bruto produkti vendor (BPV)':'GDP',
    # Trade
    'Eksporti':'Exports','Importi':'Imports',
    'Vlera Mallrave':'Value of goods (EUR)','Sasia':'Quantity',
    'Regjimi':'Customs regime','Origjina':'Origin',
    'Destinimi Final':'Destination','Kodi Tarifor':'Tariff code','Netweight':'Net weight (kg)',
    # Employment
    'PAPUNËSIA (NË MIJËRA)':'Unemployment (thousands)',
    'SHKALLA E PAPUNËSISË (%)':'Unemployment rate (%)',
    'Fuqia Punëtore (në mijëra)':'Labor force (thousands)',
    # Deposits / credit
    'Gjithsej depozitat në euro':'Total deposits (EUR mn)',
    'Depozitat e transferueshme':'Transferable deposits','Depozitat e kursimit':'Savings deposits',
    'Depozitat tjera':'Other deposits','Qeveria':'Government',
    'Norma e interesit ne kredit-Sektori Bankar':'Bank lending rates',
    # Inflation
    'Inflacioni i përgjithshëm':'Headline inflation','Inflacioni bazë':'Core inflation',
    # HBS
    'Gjithsej në Milion':'Total consumption (EUR mn)',
    'Konsumi për ekonomi familjare':'Consumption per household (EUR)',
    'Konsumi për banor':'Consumption per capita (EUR)',
    # Fiscal
    'D21 - Tatimet në produkte':'D21 - Taxes on products',
    'D211 - Tatimi mbi vleren e shtuar(TVSH)':'D211 - VAT',
    'D29 - Tatimet e tjera në prodhim':'D29 - Other taxes on production',
    'D5 - Tatimet rrjedhëse mbi të ardhura':'D5 - Income and wealth taxes',
    'D61 - Kontributet sociale neto':'D61 - Net social contributions',
    'D7 - Transfertat tjera rrjedhëse':'D7 - Other current transfers',
    # Investment
    'Investime në Ndërtim':'Construction investment','Investime në makineri':'Machinery investment',
    'Blerjet e reja':'New purchases','Blerjet e vjetra':'Second-hand purchases',
    'Investime në vijim':'Work in progress','Sektorët / Aktivitetet':'Sector / Activity',
}

def translate(s):
    if not isinstance(s, str):
        return s
    return LABEL.get(s.strip(), s.strip())

def clean_cols(df):
    df.columns = [translate(c) for c in df.columns]
    return df

def save(df, name, note=''):
    path = OUT_DIR / f'{name}.parquet'
    df.to_parquet(path, index=False)
    print(f'  saved {name}.parquet  {df.shape}  {note}')
    return df

print('Utilities loaded.')

Utilities loaded.


## 1. Remittances  *(core variable)*

In [ ]:
xl = pd.ExcelFile(RAW_KS / 'Dergesat-e-emigranteve-.xlsx')
raw = xl.parse(xl.sheet_names[0], header=None)


In [131]:
import pandas as pd
import re
def clean_data(data):
    df = data.copy()

    # First column contains years / quarters / months
    period_col = df.columns[0]
    df = df.rename(columns={period_col: "period"})

    # Translate country names from Albanian to English
    country_map = {
        "Gjithsej": "Total",
        "Gjermania": "Germany",
        "Zvicrra": "Switzerland",
        "Zvicra": "Switzerland",
        "Italia": "Italy",
        "Austria": "Austria",
        "Mbretëria e Bashkuar": "United Kingdom",
        "Danimarka": "Denmark",
        "Finlanda": "Finland",
        "Holanda": "Netherlands",
        "Sllovenija": "Slovenia",
        "Sllovenia": "Slovenia",
        "Vende tjera": "Other countries",
    }

    df = df.rename(columns=country_map)

    # Clean period strings
    df["period"] = df["period"].astype(str).str.strip()

    # Remove the standalone 2008 annual row
    df = df[df["period"] != "2008"].copy()

    # Albanian month names
    month_map = {
        "janar": 1,
        "shkurt": 2,
        "mars": 3,
        "prill": 4,
        "maj": 5,
        "qershor": 6,
        "korrik": 7,
        "gusht": 8,
        "shtator": 9,
        "tetor": 10,
        "nentor": 11,
        "nëntor": 11,
        "dhjetor": 12,
    }

    # Extract year, quarter, and month information
    df["_year_raw"] = df["period"].str.extract(r"(\d{4})")[0]
    df["_quarter"] = df["period"].str.extract(r"\bT\s*([1-4])\b")[0]

    month_regex = "|".join(month_map.keys())
    df["_month_name"] = (
        df["period"]
        .str.lower()
        .str.extract(f"({month_regex})", flags=re.IGNORECASE)[0]
    )

    df["_month"] = df["_month_name"].map(month_map)

    # Identify quarterly and monthly rows
    quarter_mask = df["_quarter"].notna()
    month_mask = df["_month"].notna()

    # Empty date column
    df["_date"] = pd.NaT

    # ---- Handle quarterly data ----
    # In your quarterly data the year appears on the T4 row, e.g. "2023 T4",
    # so we backfill the year to T1-T3.
    quarter_years = df.loc[quarter_mask, "_year_raw"].bfill().ffill()

    quarter_periods = (
        quarter_years.astype(int).astype(str)
        + "Q"
        + df.loc[quarter_mask, "_quarter"].astype(int).astype(str)
    )

    df.loc[quarter_mask, "_date"] = (
        pd.PeriodIndex(quarter_periods, freq="Q")
        .to_timestamp(how="start")
    )

    # ---- Handle monthly data ----
    # If monthly rows contain a year, use it.
    # If they do not, infer years sequentially after the last quarterly year.
    last_quarter_year = quarter_years.astype(float).max()

    current_year = int(last_quarter_year) + 1 if pd.notna(last_quarter_year) else None
    previous_month = None

    monthly_years = {}

    for idx, row in df.loc[month_mask].iterrows():
        raw_year = row["_year_raw"]
        month = int(row["_month"])

        if pd.notna(raw_year):
            current_year = int(raw_year)
        elif current_year is None:
            raise ValueError("Monthly rows have no year and no previous quarterly year to infer from.")

        # If months restart, e.g. Dhjetor -> Janar, move to next year
        if previous_month is not None and month < previous_month and pd.isna(raw_year):
            current_year += 1

        monthly_years[idx] = current_year
        previous_month = month

    monthly_years = pd.Series(monthly_years)

    df.loc[month_mask, "_date"] = pd.to_datetime({
        "year": monthly_years.astype(int),
        "month": df.loc[month_mask, "_month"].astype(int),
        "day": 1,
    })

    # Drop rows where no date could be created, if any
    df = df[df["_date"].notna()].copy()

    # Set datetime index
    df = df.set_index("_date")
    df.index.name = "date"

    # Drop helper columns
    df = df.drop(columns=[
        "period",
        "_year_raw",
        "_quarter",
        "_month_name",
        "_month",
    ])
    df.columns = [str(c) for c in df.columns]

    # Convert all data columns to numeric
    df = df.apply(pd.to_numeric, errors="coerce")

    # Optional: sort by date
    df = df.sort_index()

    data_clean = df
    return data_clean

In [135]:
# By country
data = raw.iloc[11:29].dropna(axis=1,how = "all").T
data.columns = data.iloc[0]
data = data.iloc[1:]

d = raw.iloc[42:60].dropna(axis=1,how = "all").T
d.columns = d.iloc[0]
d = d.iloc[1:]
data = pd.concat([data, d], axis=0)
data = clean_data(data)

data_2 = raw.iloc[33:38].dropna(axis=1,how = "all").T
data_2.columns = data_2.iloc[0]
data_2 = data_2.iloc[1:-1]
data_2 = pd.concat([data_2.iloc[:60], data_2.iloc[187:]], axis=0)
data_2 = clean_data(data_2)
dict = {"Gjithsej dërgesat e migrantëv": "Total","Bankat": "Banks","Agjencitë për transfer të mjeteve": "Transfer agencies","Tjera": "Other"}
data_2.rename(columns=dict, inplace=True)

for c in data.columns:
    data[c] = data[c] * data_2['Total'] / 100

In [136]:
data.to_parquet('remittance_country.parquet')
data_2.to_parquet('remittance_channel.parquet')

## 2. GDP

### By NACE

In [143]:
xl = pd.ExcelFile(RAW_KS / 'GDP-tre-mujore.xlsx')
raw = xl.parse(xl.sheet_names[1], header=None)
df = raw.iloc[2:27]

In [145]:
def clean_gdp_quarterly_by_activity(raw):
    df = raw.copy()

    # ------------------------------------------------------------
    # 1. Find the row containing quarter labels like 2025/TM2
    # ------------------------------------------------------------
    period_pattern = r"\d{4}\s*/\s*TM[1-4]"

    period_row_idx = df.apply(
        lambda row: row.astype(str).str.contains(period_pattern, regex=True).sum(),
        axis=1
    ).idxmax()

    period_row = df.loc[period_row_idx]

    # First column is the activity/sector name
    activity_col = df.columns[0]

    # Columns that contain quarter data
    quarter_cols = [
        col for col in df.columns
        if re.search(period_pattern, str(period_row[col]))
    ]

    # Keep only rows below the period row
    df = df.loc[period_row_idx + 1:, [activity_col] + quarter_cols].copy()

    # Rename columns
    df = df.rename(columns={activity_col: "activity_al"})
    df.columns = ["activity_al"] + [str(period_row[c]).strip() for c in quarter_cols]

    # Drop empty rows
    df = df[df["activity_al"].notna()].copy()

    # ------------------------------------------------------------
    # 2. Reshape wide -> long
    # ------------------------------------------------------------
    df = df.melt(
        id_vars="activity_al",
        var_name="period_raw",
        value_name="value"
    )

    # Clean values
    df["value"] = (
        df["value"]
        .replace(":", np.nan)
        .replace("", np.nan)
    )
    df["value"] = pd.to_numeric(df["value"], errors="coerce")

    # ------------------------------------------------------------
    # 3. Parse quarterly dates: 2025/TM2 -> 2025-Q2
    # ------------------------------------------------------------
    df["year"] = df["period_raw"].str.extract(r"(\d{4})").astype(int)
    df["quarter"] = df["period_raw"].str.extract(r"TM([1-4])")[0].astype(int)

    df["period"] = (
        df["year"].astype(str)
        + "-Q"
        + df["quarter"].astype(str)
    )

    df["date"] = pd.PeriodIndex(df["period"], freq="Q").to_timestamp(how="start")

    # ------------------------------------------------------------
    # 4. Replace Albanian activities with NACE / macro codes
    # ------------------------------------------------------------
    nace_map = {
        "Bujqësia, pylltaria dhe peshkimi": {
            "nace_code": "A",
            "activity_en": "Agriculture, forestry and fishing",
        },
        "Industria nxjerrëse": {
            "nace_code": "B",
            "activity_en": "Mining and quarrying",
        },
        "Industria përpunuese": {
            "nace_code": "C",
            "activity_en": "Manufacturing",
        },
        "Furnizimi me energji elektrike dhe gaz": {
            "nace_code": "D",
            "activity_en": "Electricity, gas, steam and air conditioning supply",
        },
        "Furnizimi me ujë": {
            "nace_code": "E",
            "activity_en": "Water supply; sewerage, waste management and remediation activities",
        },
        "Ndërtimtaria": {
            "nace_code": "F",
            "activity_en": "Construction",
        },
        "Tregtia me shumicë dhe pakicë": {
            "nace_code": "G",
            "activity_en": "Wholesale and retail trade; repair of motor vehicles and motorcycles",
        },
        "Transporti dhe magazinimi": {
            "nace_code": "H",
            "activity_en": "Transportation and storage",
        },
        "Hotelet dhe restorantet": {
            "nace_code": "I",
            "activity_en": "Accommodation and food service activities",
        },
        "Informimi dhe komunikimi": {
            "nace_code": "J",
            "activity_en": "Information and communication",
        },
        "Aktivitetetet financiare dhe të sigurimit": {
            "nace_code": "K",
            "activity_en": "Financial and insurance activities",
        },
        "Aktivitetet financiare dhe të sigurimit": {
            "nace_code": "K",
            "activity_en": "Financial and insurance activities",
        },
        "Afarizmi me pasuri të patundshme": {
            "nace_code": "L",
            "activity_en": "Real estate activities",
        },
        "Aktivitetet profesional, shkencore dhe teknike": {
            "nace_code": "M",
            "activity_en": "Professional, scientific and technical activities",
        },
        "Aktivitetet administrative dhe mbështetëse": {
            "nace_code": "N",
            "activity_en": "Administrative and support service activities",
        },
        "Administrimi publik dhe i mbrojtjes": {
            "nace_code": "O",
            "activity_en": "Public administration and defence; compulsory social security",
        },
        "Edukimi": {
            "nace_code": "P",
            "activity_en": "Education",
        },
        "Shëndetësia dhe aktivitetet e punës sociale": {
            "nace_code": "Q",
            "activity_en": "Human health and social work activities",
        },
        "Art, zbavitje dhe rekreacion": {
            "nace_code": "R",
            "activity_en": "Arts, entertainment and recreation",
        },
        "Shërbime tjera": {
            "nace_code": "S",
            "activity_en": "Other service activities",
        },
        "Aktivitetet e ekonomive familjare": {
            "nace_code": "T",
            "activity_en": "Activities of households as employers",
        },
        "Aktivitetet e organizatave dhe trupave ekstra": {
            "nace_code": "U",
            "activity_en": "Activities of extraterritorial organisations and bodies",
        },
        "Bruto vlera e shtuar, Gjithsej": {
            "nace_code": "GVA_TOTAL",
            "activity_en": "Gross value added, total",
        },
        "Taksta neto ne produkte": {
            "nace_code": "TAX_PRODUCTS_NET",
            "activity_en": "Net taxes on products",
        },
        "Taksa neto ne produkte": {
            "nace_code": "TAX_PRODUCTS_NET",
            "activity_en": "Net taxes on products",
        },
        "GJITHSEJ BPV": {
            "nace_code": "GDP",
            "activity_en": "Gross domestic product",
        },
    }

    def match_activity_name(x, field):
        x = str(x).strip()
        for key, val in nace_map.items():
            if x.startswith(key):
                return val[field]
        return np.nan

    df["nace_code"] = df["activity_al"].apply(lambda x: match_activity_name(x, "nace_code"))
    df["activity_en"] = df["activity_al"].apply(lambda x: match_activity_name(x, "activity_en"))

    # Optional: if you want NACE code as the main indicator
    df["indicator"] = df["nace_code"]

    # Metadata
    df["unit"] = "EUR thousand"
    df["frequency"] = "quarterly"
    df["source"] = "ASK"

    # ------------------------------------------------------------
    # 5. Final tidy dataframe with datetime index
    # ------------------------------------------------------------
    df = df[
        [
            "date",
            "indicator",
            "nace_code",
            "activity_en",
            "activity_al",
            "period",
            "year",
            "quarter",
            "value",
            "unit",
            "frequency",
            "source",
        ]
    ].copy()

    df = df.sort_values(["date", "nace_code"])
    df = df.set_index("date")

    return df


gdp_quarterly_clean = clean_gdp_quarterly_by_activity(df)

In [151]:
gdp_quarterly_clean['date'] = gdp_quarterly_clean.index
df = gdp_quarterly_clean[['indicator', 'value', 'date']].pivot(index = 'date', columns = 'indicator', values = 'value')

In [161]:
gdp_quarterly_clean.to_parquet(OUT_DIR / 'gdp_quarterly_by_activity.parquet')
df.to_parquet(OUT_DIR / 'gdp_quarterly_pivot.parquet')

### By expenditure

In [168]:
xl = pd.ExcelFile(RAW_KS / 'GDP-tre-mujore.xlsx')
raw = xl.parse(xl.sheet_names[2], header=None)
data = raw.iloc[2:15]

In [238]:
import pandas as pd
import numpy as np
import re

def clean_gdp_quarterly_expenditure(raw):
    df = raw.copy()

    # ------------------------------------------------------------
    # 1. Find the row containing quarter labels like 2025/TM2
    # ------------------------------------------------------------
    period_pattern = r"\d{4}\s*/\s*TM[1-4]"

    period_row_idx = df.apply(
        lambda row: row.astype(str).str.contains(period_pattern, regex=True).sum(),
        axis=1
    ).idxmax()

    period_row = df.loc[period_row_idx]

    # First column contains expenditure/component names
    component_col = df.columns[0]

    # Columns containing quarterly data
    quarter_cols = [
        col for col in df.columns
        if re.search(period_pattern, str(period_row[col]))
    ]

    # Keep only rows below the period row
    df = df.loc[period_row_idx + 1:, [component_col] + quarter_cols].copy()

    # Rename columns
    df = df.rename(columns={component_col: "component_al"})
    df.columns = ["component_al"] + [str(period_row[c]).strip() for c in quarter_cols]

    # Drop empty rows
    df = df[df["component_al"].notna()].copy()

    # ------------------------------------------------------------
    # 2. Reshape wide -> long
    # ------------------------------------------------------------
    df = df.melt(
        id_vars="component_al",
        var_name="period_raw",
        value_name="value"
    )

    # Clean numeric values
    df["value"] = df["value"].replace({":": np.nan, "": np.nan})
    df["value"] = pd.to_numeric(df["value"], errors="coerce")

    # ------------------------------------------------------------
    # 3. Parse quarterly dates: 2025/TM2 -> 2025-Q2
    # ------------------------------------------------------------
    df["year"] = df["period_raw"].str.extract(r"(\d{4})")[0].astype(int)
    df["quarter"] = df["period_raw"].str.extract(r"TM([1-4])")[0].astype(int)

    df["period"] = (
        df["year"].astype(str)
        + "-Q"
        + df["quarter"].astype(str)
    )

    df["date"] = pd.PeriodIndex(df["period"], freq="Q").to_timestamp(how="start")

    # ------------------------------------------------------------
    # 4. Translate components / assign clean indicator codes
    # ------------------------------------------------------------
    expenditure_map = {
        "BPV me çmime aktuale": {
            "indicator_code": "GDP_CURRENT_PRICES",
            "component_en": "GDP at current prices",
        },
        "Shpenzimet e konsumit final": {
            "indicator_code": "FINAL_CONSUMPTION_EXPENDITURE",
            "component_en": "Final consumption expenditure",
        },
        "Shpenzimet e konsumit final te ekonomive shtëpiake & IJPSHESH'": {
            "indicator_code": "HOUSEHOLD_FINAL_CONSUMPTION_EXPENDITURE",
            "component_en": "Household final consumption expenditure",
        },
        "Shpenzimet e konsumit final të ekonomive shtëpiake": {
            "indicator_code": "HOUSEHOLD_FINAL_CONSUMPTION_EXPENDITURE",
            "component_en": "Household final consumption expenditure",
        },
        "Shpenzimet e konsumit final të Qeverisë": {
            "indicator_code": "GOVERNMENT_FINAL_CONSUMPTION_EXPENDITURE",
            "component_en": "Government final consumption expenditure",
        },
        "Shpenzimet e konsumit final te Qeverisë": {
            "indicator_code": "GOVERNMENT_FINAL_CONSUMPTION_EXPENDITURE",
            "component_en": "Government final consumption expenditure",
        },
        "Formimi I bruto kapitalit": {
            "indicator_code": "GROSS_CAPITAL_FORMATION",
            "component_en": "Gross capital formation",
        },
        "Formimi i bruto kapitalit": {
            "indicator_code": "GROSS_CAPITAL_FORMATION",
            "component_en": "Gross capital formation",
        },
        "Eksporti neto": {
            "indicator_code": "NET_EXPORTS",
            "component_en": "Net exports",
        },
        "Eksporti I mallrave dhe shërbimeve": {
            "indicator_code": "EXPORTS_GOODS_SERVICES",
            "component_en": "Exports of goods and services",
        },
        "Eksporti i mallrave dhe shërbimeve": {
            "indicator_code": "EXPORTS_GOODS_SERVICES",
            "component_en": "Exports of goods and services",
        },
        "Eksporti I mallrave": {
            "indicator_code": "EXPORTS_GOODS",
            "component_en": "Exports of goods",
        },
        "Eksporti i mallrave": {
            "indicator_code": "EXPORTS_GOODS",
            "component_en": "Exports of goods",
        },
        "Eksporti I shërbimeve": {
            "indicator_code": "EXPORTS_SERVICES",
            "component_en": "Exports of services",
        },
        "Eksporti i shërbimeve": {
            "indicator_code": "EXPORTS_SERVICES",
            "component_en": "Exports of services",
        },
        "Importi I mallrave dhe shërbimeve": {
            "indicator_code": "IMPORTS_GOODS_SERVICES",
            "component_en": "Imports of goods and services",
        },
        "Importi i mallrave dhe shërbimeve": {
            "indicator_code": "IMPORTS_GOODS_SERVICES",
            "component_en": "Imports of goods and services",
        },
        "Importi i mallrave": {
            "indicator_code": "IMPORTS_GOODS",
            "component_en": "Imports of goods",
        },
        "Importi I mallrave": {
            "indicator_code": "IMPORTS_GOODS",
            "component_en": "Imports of goods",
        },
        "Importi I shërbimeve": {
            "indicator_code": "IMPORTS_SERVICES",
            "component_en": "Imports of services",
        },
        "Importi i shërbimeve": {
            "indicator_code": "IMPORTS_SERVICES",
            "component_en": "Imports of services",
        },
    }
    def clean_text(x):
        return re.sub(r"\s+", " ", str(x).strip())

    def match_component(x, field):
        x = clean_text(x)

        # Exact match first
        if x in expenditure_map:
            return expenditure_map[x][field]

        return np.nan

    df["component_al"] = df["component_al"].apply(clean_text)
    df["indicator_code"] = df["component_al"].apply(lambda x: match_component(x, "indicator_code"))
    df["component_en"] = df["component_al"].apply(lambda x: match_component(x, "component_en"))

    # Metadata
    df["unit"] = "EUR thousand"
    df["frequency"] = "quarterly"
    df["source"] = "ASK"

    # ------------------------------------------------------------
    # 5. Final tidy dataframe with datetime index
    # ------------------------------------------------------------
    df = df[
        [
            "date",
            "indicator_code",
            "component_en",
            "component_al",
            "period",
            "year",
            "quarter",
            "value",
            "unit",
            "frequency",
            "source",
        ]
    ].copy()

    df = df.sort_values(["date", "indicator_code"])
    df = df.set_index("date")
    df.index.name = "date"

    return df


gdp_expenditure_clean = clean_gdp_quarterly_expenditure(data)

In [239]:
gdp_expenditure_clean.to_parquet(
    "gdp_quarterly_by_expenditure_clean.parquet")

In [240]:
gdp_expenditure_clean['component_al'].iloc[-1]

'Shpenzimet e konsumit final te ekonomive shtëpiake & IJPSHESH'

In [241]:
gdp_expenditure_wide = (
    gdp_expenditure_clean
    .reset_index()
    .pivot(index="date", columns="indicator_code", values="value")
    .sort_index()
)

gdp_expenditure_wide.to_parquet("gdp_quarterly_by_expenditure_wide.parquet")

## 3. Balance of Payments

In [7]:
# ── Full BOP summary ──────────────────────────────────────────────────────────
xl = pd.ExcelFile(RAW_KS / 'Bilanci-i-pagesave-te-Kosoves.xlsx')
raw = xl.parse(xl.sheet_names[0], header=None)

years = [int(float(x)) for x in raw.iloc[3, 1:] if pd.notna(x)]
data  = raw.iloc[4:, :len(years)+1].copy()
data.columns = ['indicator'] + years
data = data.dropna(subset=['indicator']).copy()
data['indicator'] = data['indicator'].apply(translate)

bop = data.melt(id_vars='indicator', var_name='year', value_name='value')
bop['year']  = bop['year'].astype(int)
bop['value'] = pd.to_numeric(bop['value'].replace([':','.','-'],np.nan), errors='coerce')
bop = bop.dropna(subset=['value']).sort_values(['indicator','year']).reset_index(drop=True)
bop['unit'] = 'EUR million'
bop['frequency'] = 'annual'
bop['source'] = 'BQK'

save(bop, 'balance_of_payments')
bop[bop.indicator=='Current account'].tail(6)

  saved balance_of_payments.parquet  (819, 6)  


,indicator,year,value,unit,frequency,source
246,Current account,2023,-729.369958,EUR million,annual,BQK
247,Current account,2023,-165.279122,EUR million,annual,BQK
248,Current account,2023,114.583919,EUR million,annual,BQK
249,Current account,2024,-914.752992,EUR million,annual,BQK
250,Current account,2024,-41.276229,EUR million,annual,BQK
251,Current account,2024,96.431311,EUR million,annual,BQK


In [8]:
# ── BOP primary & secondary income (remittances channel detail) ───────────────
xl = pd.ExcelFile(RAW_KS / 'Bilanci-i-Pagesave-te-ardhurat-paresore-dhe-dytesore.xlsx')

dfs = []
for sheet in xl.sheet_names:
    raw = xl.parse(sheet, header=None)
    years = [int(float(x)) for x in raw.iloc[1, 1:] if pd.notna(x)]
    data  = raw.iloc[2:, :len(years)+1].copy()
    data.columns = ['indicator'] + years
    data = data.dropna(subset=['indicator']).copy()
    data['indicator'] = data['indicator'].apply(translate)
    data['sheet'] = sheet
    dfs.append(data)

combined = pd.concat(dfs, ignore_index=True)
bop_income = combined.melt(id_vars=['indicator','sheet'], var_name='year', value_name='value')
bop_income['year']  = bop_income['year'].astype(int)
bop_income['value'] = pd.to_numeric(bop_income['value'].replace([':','.','-'],np.nan), errors='coerce')
bop_income = bop_income.dropna(subset=['value']).sort_values(['sheet','indicator','year']).reset_index(drop=True)
bop_income['unit'] = 'EUR million'
bop_income['frequency'] = 'annual'
bop_income['source'] = 'BQK'

save(bop_income, 'bop_primary_secondary_income')
bop_income.head(6)

  saved bop_primary_secondary_income.parquet  (1407, 7)  


,indicator,sheet,year,value,unit,frequency,source
0,Asetet reservë,BIP-te ardhurat paresore,2004,0.000000,EUR million,annual,BQK
1,Asetet reservë,BIP-te ardhurat paresore,2004,0.000000,EUR million,annual,BQK
2,Asetet reservë,BIP-te ardhurat paresore,2004,-2.720000,EUR million,annual,BQK
3,Asetet reservë,BIP-te ardhurat paresore,2004,4.160000,EUR million,annual,BQK
4,Asetet reservë,BIP-te ardhurat paresore,2004,-5.491086,EUR million,annual,BQK
5,Asetet reservë,BIP-te ardhurat paresore,2004,0.178387,EUR million,annual,BQK


In [9]:
# ── Current account & goods detail ────────────────────────────────────────────
xl = pd.ExcelFile(RAW_KS / 'Llogaria-rrjedhese-dhe-mallrat.xlsx')

dfs = []
for sheet in xl.sheet_names:
    raw = xl.parse(sheet, header=None)
    years = [int(float(x)) for x in raw.iloc[4, 1:] if pd.notna(x)]
    data  = raw.iloc[5:, :len(years)+1].copy()
    data.columns = ['indicator'] + years
    data = data.dropna(subset=['indicator']).copy()
    data['indicator'] = data['indicator'].apply(translate)
    data['sheet'] = sheet
    dfs.append(data)

ca = pd.concat(dfs, ignore_index=True)
ca = ca.melt(id_vars=['indicator','sheet'], var_name='year', value_name='value')
ca['year']  = ca['year'].astype(int)
ca['value'] = pd.to_numeric(ca['value'].replace([':','.','-'],np.nan), errors='coerce')
ca = ca.dropna(subset=['value']).sort_values(['sheet','indicator','year']).reset_index(drop=True)
ca['unit'] = 'EUR million'
ca['frequency'] = 'annual'
ca['source'] = 'BQK'

save(ca, 'current_account_goods')
ca.head(4)

  saved current_account_goods.parquet  (2012, 7)  


,indicator,sheet,year,value,unit,frequency,source
0,Balance,Llogaria Rrjedhese,2004,-208.240000,EUR million,annual,BQK
1,Balance,Llogaria Rrjedhese,2004,-52.586189,EUR million,annual,BQK
2,Balance,Llogaria Rrjedhese,2004,-78.697111,EUR million,annual,BQK
3,Balance,Llogaria Rrjedhese,2005,-247.420000,EUR million,annual,BQK


## 4. Exports & Imports

In [10]:
# ── Monthly exports & imports by SITC section ─────────────────────────────────
xl = pd.ExcelFile(RAW_KS / 'Eksporti-dhe-Importi.xlsx')
raw = xl.parse('Eksporti dhe importi', header=None)

# Reconstruct two-level header: row2=period, row3=E/I direction
r2 = raw.iloc[2, 1:].ffill().tolist()    # period (forward-filled)
r3 = raw.iloc[3, 1:].tolist()            # Importi / Eksporti

def parse_ym(s):
    m = re.match(r'(\d{4})M(\d{2})', str(s))
    return (int(m.group(1)), int(m.group(2))) if m else (None, None)

cols = []
for p, d in zip(r2, r3):
    yr, mo = parse_ym(p)
    direction = translate(str(d).strip())
    cols.append((yr, mo, direction))

data = raw.iloc[4:, 1:].copy()
data.index = raw.iloc[4:, 0].apply(translate).values
data.columns = pd.MultiIndex.from_tuples(cols, names=['year','month','direction'])

ei = data.stack(level=[0,1,2]).reset_index()
ei.columns = ['section','year','month','direction','value']
ei['value'] = pd.to_numeric(ei['value'].replace([':','.','-'],np.nan), errors='coerce')
ei = ei.dropna(subset=['value','year']).copy()
ei['year']  = ei['year'].astype(int)
ei['month'] = ei['month'].astype(int)
ei = ei.sort_values(['section','year','month','direction']).reset_index(drop=True)
ei['unit'] = 'EUR thousand'
ei['frequency'] = 'monthly'
ei['source'] = 'ASK'

save(ei, 'exports_imports_monthly_by_section')
ei.head(4)

  saved exports_imports_monthly_by_section.parquet  (7842, 8)  


,section,year,month,direction,value,unit,frequency,source
0,"1.Kafshë të gjalla, produkte me origjinë kafshe",2010,1,Exports,46.0,EUR thousand,monthly,ASK
1,"1.Kafshë të gjalla, produkte me origjinë kafshe",2010,1,Imports,6312.0,EUR thousand,monthly,ASK
2,"1.Kafshë të gjalla, produkte me origjinë kafshe",2010,2,Exports,42.0,EUR thousand,monthly,ASK
3,"1.Kafshë të gjalla, produkte me origjinë kafshe",2010,2,Imports,6784.0,EUR thousand,monthly,ASK


In [11]:
# ── Customs trade balance — exports (granular product-level, 2015-2025) ────────
xl = pd.ExcelFile(RAW_KS / 'Bilanci-Tregtar-Export-Dogana-e-Kosove.xlsx')

dfs = []
for sheet in xl.sheet_names:
    if sheet.lower().startswith('sheet'):
        continue
    try:
        raw = xl.parse(sheet, header=None)
        # Find header row (contains 'VITI' or 'Viti')
        hrow = raw[raw.iloc[:,0].astype(str).str.contains('VITI|Viti', na=False)].index[0]
        df = xl.parse(sheet, header=hrow)
        df.columns = [c.strip() for c in df.columns.astype(str)]
        col_map = {'VITI':'year','Viti':'year','MUAJI':'month','Muaji':'month',
                   'Regjimi':'customs_regime','Destinimi Final':'destination',
                   'Kodi Tarifor':'tariff_code','Sasia':'quantity',
                   'Vlera Mallrave':'value_eur','Netweight':'net_weight_kg'}
        df = df.rename(columns=col_map)
        df = df[df['year'].apply(lambda x: str(x).isdigit() or (isinstance(x,(int,float)) and not np.isnan(x)))]
        df['year']  = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
        df['month'] = pd.to_numeric(df['month'], errors='coerce').astype('Int64')
        dfs.append(df)
    except Exception as e:
        print(f'  skip {sheet}: {e}')

exports_customs = pd.concat(dfs, ignore_index=True)
exports_customs['value_eur']    = pd.to_numeric(exports_customs.get('value_eur'), errors='coerce')
exports_customs['net_weight_kg']= pd.to_numeric(exports_customs.get('net_weight_kg'), errors='coerce')
exports_customs['source'] = 'Kosovo Customs'
exports_customs['direction'] = 'Exports'

save(exports_customs, 'trade_exports_customs_detail')
exports_customs.head(3)

  saved trade_exports_customs_detail.parquet  (266814, 10)  


,year,month,customs_regime,destination,tariff_code,quantity,value_eur,net_weight_kg,source,direction
0,2015,1,EX1,AL - SHQIPËRI,7204499000 - - - - - Të tjera:,NaN,682768.2,3117845.0,Kosovo Customs,Exports
1,2015,1,EX1,AL - SHQIPËRI,2306300000 - - Prej farave të lulediellit,NaN,9000.0,90000.0,Kosovo Customs,Exports
2,2015,1,EX1,AL - SHQIPËRI,8436290000 - - - Të tjera,NaN,3500.0,12000.0,Kosovo Customs,Exports


In [12]:
# ── Customs trade balance — imports (2016-2020) ────────────────────────────────
xl = pd.ExcelFile(RAW_KS / 'Bilanci-Tregtar-Importi-Dogana-e-Kosoves-2016-20.xlsx')

dfs = []
for sheet in xl.sheet_names:
    try:
        raw = xl.parse(sheet, header=None)
        hrow = raw[raw.iloc[:,0].astype(str).str.contains('VITI|Viti', na=False)].index[0]
        df = xl.parse(sheet, header=hrow)
        df.columns = [c.strip() for c in df.columns.astype(str)]
        col_map = {'VITI':'year','Viti':'year','MUAJI':'month','Muaji':'month',
                   'Regjimi':'customs_regime','Origjina':'origin',
                   'Kodi Tarifor':'tariff_code','Sasia':'quantity',
                   'Vlera Mallrave':'value_eur','Netweight':'net_weight_kg'}
        df = df.rename(columns=col_map)
        df = df[df['year'].apply(lambda x: str(x).isdigit() or (isinstance(x,(int,float)) and not np.isnan(x)))]
        df['year']  = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
        df['month'] = pd.to_numeric(df['month'], errors='coerce').astype('Int64')
        dfs.append(df)
    except Exception as e:
        print(f'  skip {sheet}: {e}')

imports_customs = pd.concat(dfs, ignore_index=True)
imports_customs['value_eur']    = pd.to_numeric(imports_customs.get('value_eur'), errors='coerce')
imports_customs['net_weight_kg']= pd.to_numeric(imports_customs.get('net_weight_kg'), errors='coerce')
imports_customs['source'] = 'Kosovo Customs'
imports_customs['direction'] = 'Imports'

save(imports_customs, 'trade_imports_customs_detail')
imports_customs.head(3)

  saved trade_imports_customs_detail.parquet  (1150252, 13)  


,year,month,customs_regime,origin,tariff_code,quantity,...,net_weight_kg,Taksa Doganës,Taksa Akcizës,Taksa TVSH-së,source,direction
0,2015,1,IM4,TR - TURQIA,9603909900 - - - - Të tjera,NaN,...,166.52,81.36,0.0,143.21,Kosovo Customs,Imports
1,2015,1,IM4,TR - TURQIA,3506910000 - - - Ngjitës të bazuar në polimere...,NaN,...,750.00,315.24,0.0,554.83,Kosovo Customs,Imports
2,2015,1,IM4,TR - TURQIA,3923900000 - - Të tjera,NaN,...,679.00,541.12,0.0,952.37,Kosovo Customs,Imports


## 5. Employment & Labor Force

In [13]:
# ── Employment ATK (administrative, monthly by sector 2019-2025) ──────────────
xl = pd.ExcelFile(RAW_KS / 'Punesimi-2019-2025-ATK.xlsx')

dfs = []
for sheet in xl.sheet_names:
    df = xl.parse(sheet, header=8)
    df.columns = [
        'year','month','sector','_drop',
        'registration_status','municipality',
        'n_taxpayers','n_employees'
    ]
    df = df.drop(columns=['_drop'], errors='ignore')
    df['year']  = pd.to_numeric(df['year'],  errors='coerce').astype('Int64')
    df['month'] = pd.to_numeric(df['month'], errors='coerce').astype('Int64')
    df['n_taxpayers'] = pd.to_numeric(df['n_taxpayers'], errors='coerce')
    df['n_employees'] = pd.to_numeric(df['n_employees'], errors='coerce')
    df = df.dropna(subset=['year','month','n_employees'])
    dfs.append(df)

employment_atk = pd.concat(dfs, ignore_index=True)
employment_atk['source'] = 'ATK'
employment_atk['frequency'] = 'monthly'
employment_atk['note'] = 'administrative register; sector x municipality x legal status'

save(employment_atk, 'employment_atk_monthly_detail')

# Aggregated version (monthly totals by sector)
emp_agg = (employment_atk
           .groupby(['year','month','sector'])[['n_taxpayers','n_employees']]
           .sum().reset_index())
save(emp_agg, 'employment_atk_monthly_by_sector')
emp_agg.head(4)

ValueError: Length mismatch: Expected axis has 30 elements, new values have 8 elements

In [ ]:
# ── Labor Force Survey — annual key indicators ────────────────────────────────
xl = pd.ExcelFile(RAW_KS / 'Anketa-e-fuqise-punetore-treguesit-vjetor.xlsx')

PRIORITY_SHEETS = [
    'Papunesia sipas gjinise',
    'Fuqia punetore',
    'Te punesuarit-Gjinia',
    'Shkalla e papunesise se te rinj',
    'Raporti punesim-popullsise',
    'Paga mujore',
]

def parse_lfs_sheet(xl, sheet):
    raw = xl.parse(sheet, header=None)
    # years are in row 2, genders in row 3; data from row 4
    # Build column tuples (year, gender)
    years_row = raw.iloc[2, 1:].tolist()
    gender_row = raw.iloc[3, 1:].tolist()
    cols = []
    cur_yr = None
    for y, g in zip(years_row, gender_row):
        if pd.notna(y) and str(y).strip():
            try: cur_yr = int(float(y))
            except: cur_yr = str(y)
        gender = translate(str(g).strip()) if pd.notna(g) else 'Total'
        cols.append((cur_yr, gender))

    data = raw.iloc[4:, :len(cols)+1].copy()
    data.columns = ['indicator'] + cols
    data = data.dropna(subset=['indicator'])
    data = data[data['indicator'].astype(str).str.strip() != ':']
    data['indicator'] = data['indicator'].apply(translate)
    data = data[data['indicator'].apply(lambda x: not str(x).startswith(':') and not str(x).startswith('Perd'))]

    records = []
    for _, row in data.iterrows():
        ind = row['indicator']
        for (yr, gen), val in zip(cols, row[1:].values):
            records.append({'indicator': ind, 'year': yr, 'gender': gen,
                            'value': pd.to_numeric(str(val).replace(':',''), errors='coerce')})
    return pd.DataFrame(records).dropna(subset=['value'])

lfs_dfs = []
for sh in PRIORITY_SHEETS:
    try:
        df = parse_lfs_sheet(xl, sh)
        df['sheet'] = sh
        lfs_dfs.append(df)
    except Exception as e:
        print(f'  skip {sh}: {e}')

lfs_annual = pd.concat(lfs_dfs, ignore_index=True)
lfs_annual['year'] = pd.to_numeric(lfs_annual['year'], errors='coerce').astype('Int64')
lfs_annual = lfs_annual.sort_values(['sheet','indicator','year','gender']).reset_index(drop=True)
lfs_annual['frequency'] = 'annual'
lfs_annual['source'] = 'ASK (LFS)'

save(lfs_annual, 'labor_force_survey_annual')
lfs_annual[lfs_annual.sheet=='Papunesia sipas gjinise'].head(6)

In [ ]:
# ── Labor Force Survey — quarterly ────────────────────────────────────────────
xl = pd.ExcelFile(RAW_KS / 'Anketa-e-fuqise-punetore-treguesit-3-mujor.xlsx')

def parse_lfs_quarterly(xl, sheet):
    raw = xl.parse(sheet, header=None)
    periods_row = raw.iloc[2, 1:].tolist()
    gender_row  = raw.iloc[3, 1:].tolist()
    cols = []
    cur_p = None
    for p, g in zip(periods_row, gender_row):
        if pd.notna(p) and str(p).strip():
            cur_p = str(p).strip()
        gender = translate(str(g).strip()) if pd.notna(g) else 'Total'
        cols.append((cur_p, gender))

    data = raw.iloc[4:, :len(cols)+1].copy()
    data.columns = ['indicator'] + cols
    data = data.dropna(subset=['indicator'])
    data['indicator'] = data['indicator'].apply(translate)

    records = []
    for _, row in data.iterrows():
        ind = row['indicator']
        for (p, gen), val in zip(cols, row[1:].values):
            if p is None: continue
            m = re.match(r'(\d{4})/(TM\d)', str(p))
            if m:
                yr, q = int(m.group(1)), QUARTER_MAP[m.group(2)]
            else:
                continue
            records.append({'indicator': ind, 'year': yr, 'quarter': q,
                            'gender': gen,
                            'value': pd.to_numeric(str(val).replace(':',''), errors='coerce')})
    return pd.DataFrame(records).dropna(subset=['value'])

lfs_q_dfs = []
for sh in xl.sheet_names:
    try:
        df = parse_lfs_quarterly(xl, sh)
        df['sheet'] = sh
        lfs_q_dfs.append(df)
    except Exception as e:
        print(f'  skip {sh}: {e}')

lfs_quarterly = pd.concat(lfs_q_dfs, ignore_index=True)
lfs_quarterly['frequency'] = 'quarterly'
lfs_quarterly['source'] = 'ASK (LFS)'

save(lfs_quarterly, 'labor_force_survey_quarterly')
lfs_quarterly.head(4)

## 6. Investment

In [ ]:
# ── Investment in enterprises (structural business survey) ────────────────────
xl = pd.ExcelFile(RAW_KS / 'Investimet-ne-nderrmarrje.xlsx')

dfs = []
for sheet in xl.sheet_names:
    try:
        raw = xl.parse(sheet, header=None)
        years = [int(float(x)) for x in raw.iloc[2, 1:] if pd.notna(x) and str(x).replace('.','').isdigit()]
        if not years: continue
        data = raw.iloc[3:, :len(years)+2].copy()
        # first two cols are category + subcategory
        data.columns = ['category','subcategory'] + years
        data['category']    = data['category'].ffill().apply(translate)
        data['subcategory'] = data['subcategory'].apply(translate)
        data = data.dropna(subset=['subcategory'])
        data['sheet'] = sheet
        dfs.append(data)
    except Exception as e:
        print(f'  skip {sheet}: {e}')

inv = pd.concat(dfs, ignore_index=True)
inv = inv.melt(id_vars=['sheet','category','subcategory'], var_name='year', value_name='value')
inv['year']  = pd.to_numeric(inv['year'], errors='coerce').astype('Int64')
inv['value'] = pd.to_numeric(inv['value'].replace([':','.','-'],np.nan), errors='coerce')
inv = inv.dropna(subset=['value']).sort_values(['sheet','category','year']).reset_index(drop=True)
inv['unit'] = 'EUR (current prices)'
inv['frequency'] = 'annual'
inv['source'] = 'ASK (SBS)'

save(inv, 'investment_enterprises')
inv[inv.sheet=='Vlera e investimeve '].head(6)

In [ ]:
# ── Foreign Direct Investment inflows ─────────────────────────────────────────
xl = pd.ExcelFile(RAW_KS / 'Investimet-e-huaja-direkte-ne-Kosove.xlsx')

dfs = []
for sheet in xl.sheet_names:
    raw = xl.parse(sheet, header=None)
    # find row with years
    for i in range(len(raw)):
        nums = [x for x in raw.iloc[i, 1:] if pd.notna(x) and str(x).replace('.','').isdigit()]
        if len(nums) > 3:
            header_row = i
            break
    years = [int(float(x)) for x in raw.iloc[header_row, 1:] if pd.notna(x) and str(x).replace('.','').isdigit()]
    data  = raw.iloc[header_row+1:, :len(years)+1].copy()
    data.columns = ['sector'] + years
    data['sector'] = data['sector'].apply(translate)
    data = data.dropna(subset=['sector'])
    data['sheet'] = sheet
    dfs.append(data)

fdi = pd.concat(dfs, ignore_index=True)
fdi = fdi.melt(id_vars=['sheet','sector'], var_name='year', value_name='value')
fdi['year']  = fdi['year'].astype(int)
fdi['value'] = pd.to_numeric(fdi['value'].replace([':','.','-'],np.nan), errors='coerce')
fdi = fdi.dropna(subset=['value']).sort_values(['sheet','sector','year']).reset_index(drop=True)
fdi['unit'] = 'EUR million'
fdi['frequency'] = 'annual'
fdi['source'] = 'BQK'

save(fdi, 'fdi_inflows')
fdi[fdi.sector=='Total'].tail(6)

## 7. Banking — Deposits & Credit

In [ ]:
# ── Deposits (monthly) ────────────────────────────────────────────────────────
xl = pd.ExcelFile(RAW_KS / 'Depozitat-Korporatat-tjera-depozituese-1-1.xlsx')
raw = xl.parse(xl.sheet_names[0], header=None)

# Row 2 = months (no year); need to reconstruct year from sheet or column sequence
# Months repeat every 12 columns
header_row = raw.iloc[2, :].tolist()   # month labels
# Find all columns by scanning for year markers (sometimes in row 0 or embedded)
# Simpler: use the full sheet with header=2 and reconstruct date from position
df_raw = xl.parse(xl.sheet_names[0], header=None)
month_labels = df_raw.iloc[2, 1:].tolist()

# Reconstruct year: data starts at what year?
# From inspection: first month is Janar, last entry visible. Count backwards from current.
# Better: read from the 'Vlera e depozitave-KTD' sheet which may have year column
raw2 = xl.parse('Vlera e depozitave-KTD', header=None)
years_row = raw2.iloc[2, 1:].tolist()
years = [int(float(x)) for x in years_row if pd.notna(x) and str(x).replace('.','').isdigit()]
data  = raw2.iloc[3:, :len(years)+1].copy()
data.columns = ['indicator'] + years
data['indicator'] = data['indicator'].apply(translate)
data = data.dropna(subset=['indicator'])

deposits = data.melt(id_vars='indicator', var_name='year', value_name='value')
deposits['year']  = deposits['year'].astype(int)
deposits['value'] = pd.to_numeric(deposits['value'].replace([':','.','-'],np.nan), errors='coerce')
deposits = deposits.dropna(subset=['value']).sort_values(['indicator','year']).reset_index(drop=True)
deposits['unit'] = 'EUR million'
deposits['frequency'] = 'annual (aggregated)'
deposits['source'] = 'BQK'

save(deposits, 'bank_deposits')
deposits.head(4)

In [ ]:
# ── Credit / lending rates (monthly from 2010) ────────────────────────────────
xl = pd.ExcelFile(RAW_KS / 'Kredite-Korporatat-tjera-depozituese-1-1.xlsx')

# Sheet 'Kredite e KTD' has annual totals
raw = xl.parse('Kredite e KTD', header=None)
years = [int(float(x)) for x in raw.iloc[2, 1:] if pd.notna(x) and str(x).replace('.','').isdigit()]
data  = raw.iloc[3:, :len(years)+1].copy()
data.columns = ['indicator'] + years
data['indicator'] = data['indicator'].apply(translate)
data = data.dropna(subset=['indicator'])

credit = data.melt(id_vars='indicator', var_name='year', value_name='value')
credit['year']  = credit['year'].astype(int)
credit['value'] = pd.to_numeric(credit['value'].replace([':','.','-'],np.nan), errors='coerce')
credit = credit.dropna(subset=['value']).sort_values(['indicator','year']).reset_index(drop=True)
credit['unit'] = 'EUR million'
credit['frequency'] = 'annual'
credit['source'] = 'BQK'

save(credit, 'bank_credit')

# Also save lending rates (monthly)
raw_rates = xl.parse('Norma e interesit ne kredi', header=None)
# Row 0 = year block header; row 1 = month labels (Jan-Dec repeated)
save(raw_rates, 'bank_lending_rates_raw')   # keep raw; already partly in English
credit.head(4)

## 8. Inflation

In [ ]:
xl = pd.ExcelFile(RAW_KS / 'Inflacioni-baze-dhe-i-pergjitshem.xlsx')
raw = xl.parse(xl.sheet_names[0], header=None)

# Row 4 = month labels (Janar…Dhjetor) with year embedded in December ('2004 Dhjetor')
header = raw.iloc[4, 1:].tolist()
indicators = raw.iloc[5:, 0].tolist()
values = raw.iloc[5:, 1:].values

# Reconstruct year-month timeline from header
dates = []
cur_year = None
cur_month = 0
for h in header:
    h_str = str(h).strip() if pd.notna(h) else ''
    # Check for year marker like '2004 Dhjetor'
    m = re.match(r'(\d{4})\s+(\S+)', h_str)
    if m:
        cur_year = int(m.group(1))
        month_name = m.group(2)
        cur_month = MONTH_AL.get(month_name, cur_month)
    elif h_str in MONTH_AL:
        cur_month = MONTH_AL[h_str]
        if cur_month == 1 and cur_year is not None:
            cur_year += 1
    dates.append((cur_year, cur_month))

records = []
for i, ind in enumerate(indicators):
    if not isinstance(ind, str) or not ind.strip():
        continue
    ind_en = translate(ind)
    for j, (yr, mo) in enumerate(dates):
        if yr is None: continue
        try:
            val = float(values[i, j])
            records.append({'indicator': ind_en, 'year': yr, 'month': mo, 'value': val})
        except:
            pass

inflation = pd.DataFrame(records).sort_values(['indicator','year','month']).reset_index(drop=True)
inflation['unit'] = '% (month-on-month)'
inflation['frequency'] = 'monthly'
inflation['source'] = 'BQK'

save(inflation, 'inflation')
inflation[inflation.indicator=='Headline inflation'].tail(8)

## 9. Household Budget Survey & Income / Living Conditions

In [ ]:
# ── HBS — key consumption sheets ─────────────────────────────────────────────
xl = pd.ExcelFile(RAW_KS / 'Anketa-e-buxhetit-te-familjes.xlsx')

KEEP_SHEETS = [
    'Konsumi i përgjithshëm Kosovë ',
    'Shpërndarja e konsumit ',
    'Konsumi vjetor i eko-lokaliteti',
    'Konsumi vjetor-viti',
    'Burimi kryesor i të hyrave',
    'Të ardhurat mesatare vjetore',
    'Të ardhurat mesatare',
]

hbs_dfs = []
for sheet in KEEP_SHEETS:
    if sheet not in xl.sheet_names:
        continue
    raw = xl.parse(sheet, header=None)
    years = [int(float(x)) for x in raw.iloc[2, 1:] if pd.notna(x) and str(x).replace('.0','').isdigit()]
    if not years: continue
    data = raw.iloc[3:, :len(years)+1].copy()
    data.columns = ['indicator'] + years
    data = data.dropna(subset=['indicator'])
    data = data[~data['indicator'].astype(str).str.startswith(('<','Simbol',':'))]
    data['indicator'] = data['indicator'].apply(translate)
    data['sheet'] = sheet
    hbs_dfs.append(data)

hbs = pd.concat(hbs_dfs, ignore_index=True)
hbs = hbs.melt(id_vars=['sheet','indicator'], var_name='year', value_name='value')
hbs['year']  = hbs['year'].astype(int)
hbs['value'] = pd.to_numeric(hbs['value'].replace([':','.','-'],np.nan), errors='coerce')
hbs = hbs.dropna(subset=['value']).sort_values(['sheet','indicator','year']).reset_index(drop=True)
hbs['frequency'] = 'annual'
hbs['source'] = 'ASK (HBS)'

save(hbs, 'household_budget_survey')
hbs[hbs.indicator.str.contains('Total|Household|capita', case=False)].tail(6)

In [ ]:
# ── Income and Living Conditions Survey ──────────────────────────────────────
xl = pd.ExcelFile(RAW_KS / 'Anketa-mbi-te-ardhurat-dhe-kushtet-e-jeteses.xlsx')

ilcs_dfs = []
for sheet in xl.sheet_names:
    try:
        raw = xl.parse(sheet, header=None)
        years = [int(float(x)) for x in raw.iloc[2, 1:] if pd.notna(x) and str(x).replace('.0','').isdigit()]
        if not years: continue
        data = raw.iloc[3:, :len(years)+1].copy()
        data.columns = ['category'] + years
        data = data.dropna(subset=['category'])
        data = data[~data['category'].astype(str).str.startswith(('<','Simbol',':','\n'))]
        data['category'] = data['category'].apply(translate)
        data['sheet'] = sheet
        ilcs_dfs.append(data)
    except:
        pass

ilcs = pd.concat(ilcs_dfs, ignore_index=True)
ilcs = ilcs.melt(id_vars=['sheet','category'], var_name='year', value_name='value')
ilcs['year']  = ilcs['year'].astype(int)
ilcs['value'] = pd.to_numeric(ilcs['value'].replace([':','.','-'],np.nan), errors='coerce')
ilcs = ilcs.dropna(subset=['value']).sort_values(['sheet','category','year']).reset_index(drop=True)
ilcs['frequency'] = 'annual'
ilcs['source'] = 'ASK (SILC)'

save(ilcs, 'income_living_conditions')
ilcs.head(4)

## 10. Wages

In [ ]:
# ── Average gross & net wages ATK (monthly by sector/gender/age) ─────────────
xl = pd.ExcelFile(RAW_KS / 'Niveli-i-pages-mesatare-bruto-dhe-neto-ATK.xlsx')

wage_dfs = []
for sheet in xl.sheet_names:
    try:
        df = xl.parse(sheet, header=8)
        # Drop all-NaN rows and the URL metadata rows at top
        df = df.dropna(how='all')
        # Keep only rows that start with a numeric YEAR
        if df.empty: continue
        first_col = df.columns[0]
        df = df[pd.to_numeric(df[first_col], errors='coerce').notna()]
        df = df.rename(columns={df.columns[0]:'year', df.columns[1]:'month'})
        df['year']  = pd.to_numeric(df['year'],  errors='coerce').astype('Int64')
        df['month'] = pd.to_numeric(df['month'], errors='coerce').astype('Int64')
        df['sheet_year'] = sheet
        wage_dfs.append(df)
    except Exception as e:
        print(f'  skip {sheet}: {e}')

wages_raw = pd.concat(wage_dfs, ignore_index=True)
wages_raw['source'] = 'ATK'
wages_raw['frequency'] = 'monthly'
wages_raw['unit'] = 'EUR'

save(wages_raw, 'wages_gross_net_monthly')
wages_raw.head(3)

## 11. Government Fiscal

In [ ]:
xl = pd.ExcelFile(RAW_KS / 'Te-hyrat-dhe-shpenzimet-qeveritare-vjetore.xlsx')

fiscal_dfs = []
for sheet in xl.sheet_names:
    raw = xl.parse(sheet, header=None)
    years = [int(float(x)) for x in raw.iloc[2, 1:] if pd.notna(x) and str(x).replace('.0','').isdigit()]
    if not years: continue
    data = raw.iloc[3:, :len(years)+1].copy()
    data.columns = ['indicator'] + years
    data = data.dropna(subset=['indicator'])
    data['indicator'] = data['indicator'].apply(translate)
    data['sheet'] = sheet
    fiscal_dfs.append(data)

fiscal = pd.concat(fiscal_dfs, ignore_index=True)
fiscal = fiscal.melt(id_vars=['sheet','indicator'], var_name='year', value_name='value')
fiscal['year']  = fiscal['year'].astype(int)
fiscal['value'] = pd.to_numeric(fiscal['value'].replace([':','.','-'],np.nan), errors='coerce')
fiscal = fiscal.dropna(subset=['value']).sort_values(['sheet','indicator','year']).reset_index(drop=True)
fiscal['unit'] = 'EUR million'
fiscal['frequency'] = 'annual'
fiscal['source'] = 'Ministry of Finance'

save(fiscal, 'government_fiscal_annual')
fiscal.head(4)

## 12. Balkans comparative data

In [ ]:
# ── IMF Balance of Payments — all Balkans countries ───────────────────────────
xl = pd.ExcelFile(RAW_BLK / 'Balance-of-Payments-BOP-IMF.xlsx')
raw = xl.parse(xl.sheet_names[0], header=None)

years = [int(float(x)) for x in raw.iloc[3, 5:] if pd.notna(x) and str(x).replace('.','').isdigit()]
header = raw.iloc[3, :5].tolist() + years
data = raw.iloc[4:, :5+len(years)].copy()
data.columns = ['country','accounting_entry','indicator','unit','scale'] + years
data = data.dropna(subset=['country'])

bop_blk = data.melt(id_vars=['country','accounting_entry','indicator','unit','scale'],
                    var_name='year', value_name='value')
bop_blk['year']  = bop_blk['year'].astype(int)
bop_blk['value'] = pd.to_numeric(bop_blk['value'].replace([':','.','-'],np.nan), errors='coerce')
bop_blk = bop_blk.dropna(subset=['value']).sort_values(['country','indicator','year']).reset_index(drop=True)
bop_blk['source'] = 'IMF BOP'
bop_blk['frequency'] = 'annual'

save(bop_blk, 'balkans_bop_imf')
bop_blk[bop_blk.country.str.contains('Kosovo',na=False)].head(4)

In [ ]:
# ── World Bank GDP — Balkans ──────────────────────────────────────────────────
xl = pd.ExcelFile(RAW_BLK / 'GDP-1.xlsx')
raw = xl.parse('GDP', header=None)

years = [int(float(x)) for x in raw.iloc[2, 3:] if pd.notna(x) and str(x).replace('.','').isdigit()]
data  = raw.iloc[3:, :3+len(years)].copy()
data.columns = ['country','country_code','series'] + years
data = data.dropna(subset=['country'])

gdp_blk = data.melt(id_vars=['country','country_code','series'], var_name='year', value_name='value')
gdp_blk['year']  = gdp_blk['year'].astype(int)
gdp_blk['value'] = pd.to_numeric(gdp_blk['value'].replace([':','.','-'],np.nan), errors='coerce')
gdp_blk = gdp_blk.dropna(subset=['value']).sort_values(['country','series','year']).reset_index(drop=True)
gdp_blk['source'] = 'World Bank WDI'
gdp_blk['frequency'] = 'annual'

save(gdp_blk, 'balkans_gdp_wb')
gdp_blk[gdp_blk.country=='Kosovo'].head(4)

In [ ]:
# ── Remittance cost prices — World Bank ───────────────────────────────────────
for fname, label in [
    ('Remittance_Prices_Worldwide_Receiving_Countries-World-Bank.xlsx', 'receiving'),
    ('Remittance_Prices_Worldwide_Sending_Countries-World-Bank.xlsx',   'sending'),
]:
    xl = pd.ExcelFile(RAW_BLK / fname)
    raw = xl.parse(xl.sheet_names[0], header=None)
    years = [int(float(x)) for x in raw.iloc[2, 1:] if pd.notna(x) and str(x).replace('.','').isdigit()]
    if not years:
        # try header=2
        df = xl.parse(xl.sheet_names[0], header=2)
        df.to_parquet(OUT_DIR / f'balkans_remittance_prices_{label}.parquet', index=False)
        print(f'  saved balkans_remittance_prices_{label}.parquet (raw)  {df.shape}')
        continue
    data = raw.iloc[3:, :len(years)+1].copy()
    data.columns = ['country'] + years
    data = data.dropna(subset=['country'])
    df = data.melt(id_vars='country', var_name='year', value_name='value')
    df['year']  = df['year'].astype(int)
    df['value'] = pd.to_numeric(df['value'].replace([':','.','-'],np.nan), errors='coerce')
    df = df.dropna(subset=['value']).sort_values(['country','year']).reset_index(drop=True)
    df['direction'] = label
    df['source'] = 'World Bank RPW'
    save(df, f'balkans_remittance_prices_{label}')

## 13. Classification Summary

In [ ]:
classification = pd.DataFrame([
  # file                                     channel                                    relevance  freq       start  end    geography  source
  ('remittances',                            'Remittances (core)',                      '★★★★★',   'Annual',  2004,  2024,  'Kosovo',  'BQK'),
  ('balance_of_payments',                    'BOP / external stability',                '★★★★★',   'Annual',  2004,  2024,  'Kosovo',  'BQK'),
  ('bop_primary_secondary_income',           'BOP / remittances & compensation',        '★★★★★',   'Annual',  2004,  2024,  'Kosovo',  'BQK'),
  ('current_account_goods',                  'Trade balance / current account',         '★★★★☆',   'Annual',  2004,  2024,  'Kosovo',  'BQK'),
  ('gdp_annual_by_activity',                 'GDP / output growth',                     '★★★★★',   'Annual',  2000,  2024,  'Kosovo',  'ASK'),
  ('gdp_quarterly_by_activity',              'GDP / output growth',                     '★★★★★',   'Quarterly',2010, 2025,  'Kosovo',  'ASK'),
  ('exports_imports_monthly_by_section',     'Consumption → Imports',                   '★★★★★',   'Monthly', 2014,  2025,  'Kosovo',  'ASK'),
  ('trade_exports_customs_detail',           'Trade / product composition',             '★★★☆☆',   'Monthly', 2015,  2025,  'Kosovo',  'Kosovo Customs'),
  ('trade_imports_customs_detail',           'Trade / product composition',             '★★★☆☆',   'Monthly', 2016,  2020,  'Kosovo',  'Kosovo Customs'),
  ('employment_atk_monthly_by_sector',       'Employment / labor market',               '★★★★★',   'Monthly', 2019,  2025,  'Kosovo',  'ATK'),
  ('employment_atk_monthly_detail',          'Employment (firm-level detail)',           '★★★☆☆',   'Monthly', 2019,  2025,  'Kosovo',  'ATK'),
  ('labor_force_survey_annual',              'Labor force participation',               '★★★★★',   'Annual',  2012,  2024,  'Kosovo',  'ASK LFS'),
  ('labor_force_survey_quarterly',           'Labor force participation',               '★★★★☆',   'Quarterly',2015, 2025,  'Kosovo',  'ASK LFS'),
  ('investment_enterprises',                 'Savings → Investment → Capital formation','★★★★☆',   'Annual',  2018,  2023,  'Kosovo',  'ASK SBS'),
  ('fdi_inflows',                            'Capital formation / FDI',                 '★★★★☆',   'Annual',  2007,  2024,  'Kosovo',  'BQK'),
  ('bank_deposits',                          'Remittances → bank deposits → credit',    '★★★★☆',   'Annual',  2005,  2024,  'Kosovo',  'BQK'),
  ('bank_credit',                            'Credit growth → investment/consumption',  '★★★★☆',   'Annual',  2005,  2024,  'Kosovo',  'BQK'),
  ('bank_lending_rates_raw',                 'Credit channel / monetary transmission',  '★★★☆☆',   'Monthly', 2010,  2025,  'Kosovo',  'BQK'),
  ('inflation',                              'Price level / purchasing power',          '★★★★☆',   'Monthly', 2004,  2025,  'Kosovo',  'BQK'),
  ('household_budget_survey',                'Consumption / household spending',        '★★★★★',   'Annual',  2011,  2022,  'Kosovo',  'ASK HBS'),
  ('income_living_conditions',               'Income / welfare (SILC)',                 '★★★★☆',   'Annual',  2017,  2023,  'Kosovo',  'ASK SILC'),
  ('wages_gross_net_monthly',                'Wages / labor income',                    '★★★★☆',   'Monthly', 2021,  2025,  'Kosovo',  'ATK'),
  ('government_fiscal_annual',               'Fiscal multiplier / transfers',           '★★★☆☆',   'Annual',  2018,  2024,  'Kosovo',  'MoF'),
  ('balkans_bop_imf',                        'Regional BOP comparison',                 '★★★☆☆',   'Annual',  2005,  2023,  'Balkans', 'IMF'),
  ('balkans_gdp_wb',                         'Regional GDP comparison',                 '★★★☆☆',   'Annual',  2000,  2023,  'Balkans', 'World Bank'),
  ('balkans_remittance_prices_receiving',    'Remittance cost / channel efficiency',    '★★★☆☆',   'Annual',  2010,  2022,  'Balkans', 'WB RPW'),
  ('balkans_remittance_prices_sending',      'Remittance cost / sending countries',     '★★★☆☆',   'Annual',  2010,  2022,  'Balkans', 'WB RPW'),
], columns=['dataset','channel','relevance','frequency','start_year','end_year','geography','source'])

classification = classification.sort_values('relevance', ascending=False)

classification.to_parquet(OUT_DIR / 'dataset_classification.parquet', index=False)
classification.to_csv(OUT_DIR / 'dataset_classification.csv', index=False)

print('\n=== Dataset Classification ===')
display(classification.reset_index(drop=True))

In [ ]:
# Summary of all saved parquet files
print('\n=== Parquet files written ===')
for p in sorted(OUT_DIR.glob('*.parquet')):
    df = pd.read_parquet(p)
    print(f'{p.name:<55} {str(df.shape):<18}')